# Preprocessing for using `Boltz-2` conditioned on Bayesian likelihoods
This notebook contains two sections for preparing `Boltz-2` to run with Bayesian likelihoods, or equivalently, collective variable biases. The first section allows a user to locate which atom indices belong to the residues which should participate in the likelihood/colvar. These atom indices are what is required for defining a selection within the `src/boltz/colvars/colvars.py` file. The second section generates a pytorch tensor called `connected_atom_index.pt` which stores the atom indices for the C-N polypeptide bond along the backbone. At runtime, when the `--use_potentials` and `--use_colvars` flags are passed, our customized Boltz-2, on its own without further action by a user, attempts to load this file from the parent Boltz-2 directory. When it does so, it adds these atoms into the Boltz-2 `ConnectionsPotential` to enforce that the backbone should not stretch to unphysical distances.

In both sections, the user is expected to modify a `Inputs to be specified by user` section. Further information is to be found in the markdown cells belonging to each section.

#### Load the necessary libraries

In [ ]:
import torch
import multiprocessing
from pathlib import Path

In [ ]:
from boltz.data.module.inferencev2 import Boltz2InferenceDataModule
from boltz.data.types import MSA, Manifest, Record

In [ ]:
from boltz.main import filter_inputs_structure, BoltzProcessedInput, process_inputs, check_inputs
from boltz.data.module.inferencev2 import load_input

## Section 1: atom id for `Boltz-2` colvars
This section of the notebook recreates the processed structure files from the Boltz input data. This allows users to find the atom id numbers to which they want to define their collective variables, ie functions of the atomic coordinates which are to be constrained. The list the user should inspect is in the `Output for user to inspect: residues by atom list` cells just before `Section 2`.

### Inputs to be specified by user

In [ ]:
data = "/home/sotomayor/Documents/klauscj/boltz/titin/1tit.fasta"
out_dir = "/home/sotomayor/Documents/klauscj/boltz/boltz_results_1tit"

### Inputs that usually stay constant

In [ ]:
cache   = "~/.boltz/boltz2"
use_msa_server = True
msa_server_url = "https://api.colabfold.com"
msa_pairing_strategy = "greedy"
msa_server_username = None
msa_server_password = None
api_key_header = None
api_key_value = None
boltz2 = True
preprocessing_threads = multiprocessing.cpu_count()
max_msa_seqs = 8192
override = False
num_workers = 2
method = None
affinity = False

### Generate the structures for atom extraction
#### Generate the files

In [ ]:
# Set cache path
cache = Path(cache).expanduser()
cache.mkdir(parents=True, exist_ok=True)

In [ ]:
# Create output directories
data = Path(data).expanduser()
out_dir = Path(out_dir).expanduser()
out_dir = out_dir / f"boltz_results_{data.stem}"
out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Validate inputs
data = check_inputs(data)

In [ ]:
# Process inputs
ccd_path = cache / "ccd.pkl"
mol_dir = cache / "mols"

process_inputs(
        data=data,
        out_dir=out_dir,
        ccd_path=ccd_path,
        mol_dir=mol_dir,
        use_msa_server=use_msa_server,
        msa_server_url=msa_server_url,
        msa_pairing_strategy=msa_pairing_strategy,
        msa_server_username=msa_server_username,
        msa_server_password=msa_server_password,
        api_key_header=api_key_header,
        api_key_value=api_key_value,
        boltz2=boltz2,
        preprocessing_threads=preprocessing_threads,
        max_msa_seqs=max_msa_seqs,
)

In [ ]:
# Load manifest
manifest = Manifest.load(out_dir / "processed" / "manifest.json")

# Filter out existing predictions
filtered_manifest = filter_inputs_structure(
    manifest=manifest,
    outdir=out_dir,
    override=override
)

In [ ]:
# Load processed data
processed_dir = out_dir / "processed"
processed = BoltzProcessedInput(
        manifest=filtered_manifest,
        targets_dir=processed_dir / "structures",
        msa_dir=processed_dir / "msa",
        constraints_dir=(
            (processed_dir / "constraints")
            if (processed_dir / "constraints").exists()
            else None
        ),
        template_dir=(
            (processed_dir / "templates")
            if (processed_dir / "templates").exists()
            else None
        ),
        extra_mols_dir=(
            (processed_dir / "mols") if (processed_dir / "mols").exists() else None
        ),
)

In [ ]:
# Create data module
data_module = Boltz2InferenceDataModule(
                manifest=processed.manifest,
                target_dir=processed.targets_dir,
                msa_dir=processed.msa_dir,
                mol_dir=mol_dir,
                num_workers=num_workers,
                constraints_dir=processed.constraints_dir,
                template_dir=processed.template_dir,
                extra_mols_dir=processed.extra_mols_dir,
                override_method=method,
)
dataset = data_module.predict_dataloader().dataset

In [ ]:
record = dataset.manifest.records[0]

In [ ]:
input_data = load_input(
            record=record,
            target_dir=processed.targets_dir,
            msa_dir=processed.msa_dir,
            constraints_dir=processed.constraints_dir,
            template_dir=processed.template_dir,
            extra_mols_dir=processed.extra_mols_dir,
            affinity=affinity,
)

In [ ]:
tokenized = dataset.tokenizer.tokenize(input_data)

### Output for user to inspect: residues by atom list

In [ ]:
input_data.structure.residues

## Section 2: export a connectivity list
We export an index list which connects the C-terminal backbone atom within a residue to the N-initial backbone atom within the next residue. Because the polypeptide bond runs along a single chain, the user must enter the starting residue index of each chain in the cell below.

## Inputs to be specified by user

In [ ]:
# User enter list of each chain's first residue
firstresid_chains = [0]

## Generate the per chain connectivity lists along the polypeptide bond N and C residue terminii
**Note**: If there are ligands in the structure prediction, it is recommended that the sequence's `fasta` or `yaml` be written with those entities at the final chain ids, so that they are easy to not include in this connectivity list. In that case, the `firstresid_chains.append(len(input_data.structure.residues))` entry in the cell below should have the `len(input_data.structure.residues)` argument modified to be the length `input_data.structure.residues` over the residues that only contain protein. 

**Note**: If there is dna or rna in the prediction, we recommend the user not use these cells for those entities but instead rewrite the cell to enforce a more appropriate analog of a backbone potential for these biomolecules.

In [ ]:
nchains = len(firstresid_chains)
firstresid_chains.append(len(input_data.structure.residues))

In [ ]:
connected_atom_index = []
for k in range(nchains):
    connected_atom_list_C = torch.tensor([input_data.structure.residues[idend]["atom_center"]+1 
                                          for idend in range(firstresid_chains[k],firstresid_chains[k+1]-1)])
    connected_atom_list_N = torch.tensor([input_data.structure.residues[idbeg]["atom_idx"] 
                                          for idbeg in range(firstresid_chains[k]+1,firstresid_chains[k+1])])
    chain_connected_atom_list = torch.hstack((connected_atom_list_C.reshape((-1,1)),connected_atom_list_N.reshape((-1,1))))
    chain_connected_atom_list = chain_connected_atom_list.t()
    connected_atom_index.append(chain_connected_atom_list)
connected_atom_index = torch.hstack(connected_atom_index)
connected_atom_index = connected_atom_index.unsqueeze(0)

In [ ]:
torch.save(connected_atom_index,"../../connected_atom_index.pt")